# 06 — Conformal Prediction for LLMs

Conformal prediction provides coverage guarantees on LLM outputs with minimal assumptions.

## Coverage Guarantees and Prediction Sets

**Conformal prediction** provides a formal coverage guarantee: the prediction set contains the true answer with probability ≥ 1-α, for any user-specified error rate α.

For classification:
1. Compute nonconformity scores on a held-out calibration set: $s_i = 1 - \hat{p}(y_i|x_i)$
2. Set the threshold *q* at the (1-α)(1+1/n) quantile of calibration scores
3. At test time, the prediction set is: $C(x) = \{y : 1 - \hat{p}(y|x) \leq q\}$

This guarantees: $P(y_{test} \in C(x_{test})) \geq 1 - \alpha$ under exchangeability.

**Conformal risk control** (Angelopoulos et al., 2022) generalises this to LLM outputs where 'coverage' is user-defined (e.g., 'the generated answer is correct'). The risk function replaces the binary coverage indicator:
$$\hat{R}(\lambda) = \frac{1}{n} \sum_i \ell(f_\lambda(x_i), y_i) \leq \alpha$$
The threshold λ is chosen so that the average loss is bounded with high probability.

**Practical applications for LLMs**:
- QA with prediction sets: return a set of candidate answers guaranteed to contain the correct one
- Abstention: set a confidence threshold below which the model abstains
- RAG: include a variable number of retrieved chunks until coverage is reached

In [ ]:
# Conformal risk control for QA systems
import numpy as np
from scipy.stats import binom

np.random.seed(42)

# Simulate QA system with confidence scores
class QASystem:
    def predict(self, question_id):
        """Returns (confidence, is_correct) for a given question."""
        np.random.seed(question_id)
        conf = np.random.beta(3, 2)  # biased toward higher confidence
        # Model is better when confident
        is_correct = np.random.random() < (0.3 + 0.6 * conf)
        return conf, int(is_correct)

qa = QASystem()

# Step 1: Calibration set
n_cal = 300
cal_confs = []
cal_corrects = []
for i in range(n_cal):
    conf, correct = qa.predict(i)
    cal_confs.append(conf)
    cal_corrects.append(correct)

cal_confs = np.array(cal_confs)
cal_corrects = np.array(cal_corrects)

# Nonconformity scores: 1 - confidence when wrong, 0 when correct
# For coverage guarantee: conformal abstention
alpha = 0.1  # 90% coverage

# Nonconformity score: 1 - confidence (high = nonconforming)
scores = 1 - cal_confs

# Threshold: (1-alpha) quantile of calibration scores
q = np.quantile(scores, (1 - alpha) * (1 + 1/n_cal))
print(f'Conformal threshold q={q:.4f} at alpha={alpha}')

# Step 2: Test set
n_test = 500
test_results = []
for i in range(n_cal, n_cal + n_test):
    conf, correct = qa.predict(i)
    in_set = (1 - conf) <= q  # if score <= threshold, answer is in prediction set
    test_results.append({'conf': conf, 'correct': correct, 'in_set': in_set})

# Coverage: fraction of correct answers that are included
covered = [r['correct'] and r['in_set'] for r in test_results]
total_correct = sum(r['correct'] for r in test_results)
coverage = sum(r['in_set'] and r['correct'] for r in test_results) / max(total_correct, 1)

# Efficiency: fraction of questions we answer (not abstain)
efficiency = sum(r['in_set'] for r in test_results) / n_test

print(f'\nTest set results:')
print(f'  Coverage (of correct answers): {coverage:.3f} (target: {1-alpha:.1f})')
print(f'  Efficiency (fraction answered): {efficiency:.3f}')
print(f'  Abstained questions: {n_test - sum(r["in_set"] for r in test_results)}')

In [ ]:
# Visualise coverage vs threshold
import matplotlib.pyplot as plt

thresholds = np.linspace(0, 1, 50)
coverages, efficiencies = [], []

for thresh in thresholds:
    in_set = [r['in_set'] for r in test_results if (1 - r['conf']) <= thresh]
    cov_at_thresh = sum(1 for r in test_results
                        if (1 - r['conf']) <= thresh and r['correct']) / max(total_correct, 1)
    eff_at_thresh = sum(1 for r in test_results if (1 - r['conf']) <= thresh) / n_test
    coverages.append(cov_at_thresh)
    efficiencies.append(eff_at_thresh)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(thresholds, coverages, color='steelblue', label='Coverage')
ax1.axhline(1 - alpha, color='tomato', linestyle='--', label=f'Target {1-alpha:.0%}')
ax1.axvline(q, color='green', linestyle='--', label=f'Chosen q={q:.2f}')
ax1.set_xlabel('Nonconformity threshold q')
ax1.set_ylabel('Coverage')
ax1.set_title('Conformal Coverage vs Threshold')
ax1.legend()

ax2.plot(efficiencies, coverages, color='purple')
ax2.axhline(1 - alpha, color='tomato', linestyle='--', label=f'Target {1-alpha:.0%}')
ax2.set_xlabel('Efficiency (fraction answered)')
ax2.set_ylabel('Coverage')
ax2.set_title('Coverage vs Efficiency Tradeoff')
ax2.legend()

plt.tight_layout()
plt.savefig('/tmp/conformal_llm.png', dpi=100, bbox_inches='tight')
plt.show()
print('Conformal prediction for LLMs saved')